In [ ]:
!pip install openai

In [ ]:
!unzip your_data.zip

In [ ]:
#configuration
client = OpenAI(
    api_key='API-KEY', # Change to your active API KEY
    base_url="https://api.deepseek.com")

folder_name = "your_data" # Change to the path of the folder containing works to be annotated

In [ ]:
# File name: DeepSeek_Annotator.ipynb
# This program allows you to connect to the DeepSeek API to annotate works of fanfiction
# Author: Maxim van der Maesen de Sombreff
# Date: 29-06-2025

import os
from openai import OpenAI
import re
import pandas as pd
import json

"""
This script processes containing works of fanfiction by prompting DeepSeek using its API.
The prompt task the LLM to analyze the works and identify instances of non-consensual sexual material
and then categorize each instance based on 'explicitness' (Low/Medium/High) and 'focalization'
(Victim/Outsider/Perpetrator).

The JSON outputs are then formatted and exported to 'output.csv'
"""

files_to_process = [f for f in os.listdir(folder_name) if f.endswith('.txt')]

output = []
print(f"Starting annotation for {len(files_to_process)} files...")
process_number = 1

for f in files_to_process:
    print(f"Processing file {process_number} of {len(files_to_process)}: {f}")
    process_number += 1
    full_path = os.path.join(folder_name, f)
    with open(full_path, 'r', encoding='utf-8') as file:
        text_content = file.read()

    response = client.chat.completions.create(
        model="deepseek-v4-pro",
        messages=[
            {
                "role": "system",
                "content": """You are an expert in the analysis of works of fanfiction. Your role is deciding the explicity and focalization of a work of fanfiction that contains rape/non-con material.

                ===definitions===
                The labels you will choose from are High/Medium/Low for explicity.
                Low explicitness refers to allusions to rape/non-con without mention of physical events (e.g., “Achilles might be gentle with her, yet it was still rape, not love.”). This includes metaphorical statements
                Medium explicitness includes direct mentions of rape/non-con but without detailed description (e.g., “It seems that my father rapes anything that moves. No one of my family has ever rebelled against him, yet he forced
                himself on my mother Alcmene, and this is how I was conceived.”)
                High explicitness covers explicit mention of events, physical acts, and/or physical and psychological effects (e.g., “I could hear my bones grinding together where he held my wrists, and it hurt. From the waist down it hurt
                so much, like I had been torn in half.”).

                The labels you will choose from are Victim/Outsider/Perpetrator for focalization.
                Victim focalization is when the focus is on the character who is harmed physically and/or emotionally in the event.
                Outsider focalization is when the story mainly narrates the experiences and perceptions of the character who is causing another character harm.
                Perpetrator focalization is when either no experiences, character-internal emotional states or perceptions are narrated.

                ===approach===
                Use the following approach when deciding on the explicity of the work.
                -step 1: Analyze the text and identify the sentences in which rape or the act of rape is described.
                -step 2: Label the explicity of each sentence based on the analysis.
                -step 3: Identify the victim and perpetrator in the sentence.
                -step 4: Label the focalization based on the analysis.
                -step 5: If you believe no text about rape is present, label the text as "Low".
                -step 6: If you think the text is not written trough the perspective of either victim or perpetrator, label the text as "Outsider".

                ===important===
                If a scene has multiple sentences or phrases of the act beign described, split up the scene if there is too much text in between the sentences of text
                describing rape. The analyzed instances should be longer than a few senctences at most. DO NOT COPY ENTIRE PARAGRAPHS INTO YOUR OUTPUT.

                ===Examples===
                "I was raped." victim, medium.
                "Only then Prometheus realized that, when the punishment would exhaust his body and make him lose consciousness permanently and stop existing as a sentient being, Zeus could pronounce him dead, and force himself on Pronoia." outsider, low
                "He yanked Penelope forward into his arms and clasped her tight, creasing and bunching the fragile linen of her gown.  Eurynomus heard a soft feminine gasp at the manhandling.  The small sound should have given him pause, made his heart cringe away.  But instead, something lower stirred." perpetrator high


               """
            },
            {"role": "user", "content": f"""===Instructions===
              Analysize this work {text_content}.
              For this work of fanfiction first write the sentences you analyzed, then your reasoning, then the label you gave it. Output as JSON:
              {{ "instance": "<the analyzed sentences>", "reasoning_focalization": "<your reasoning>", "label_focalization": "<your verdict>", "reasoning_explicity": "<your reasoning>", "label_explicity": "<your verdict> }}

              Be sure do this process for all sentences you think contain rape/non-con. Be sure to copy the exact text you analyzed for the sentence(s) value in the JSON format.
              if multiple instances are present format your output like this:
              {{
                "instance":
                "reasoning_focalization":
                "label_focalization":
                "reasoning_explicity":
                "label_explicity":
              }},
              {{
                "instance":
                "reasoning_focalization":
                "label_focalization":
                "reasoning_explicity":
                "label_explicity":
              }}"""},
        ],
        stream=False,
        reasoning_effort="max",
        extra_body={"thinking": {"type": "enabled"}}
    )

    txt = response.choices[0].message.content
    print(txt)

    #Parsing structure for multiple JSON output formats
    try:
        parsed_data = json.loads(txt)

        if isinstance(parsed_data, dict) and "instances" in parsed_data:
            scenes_list = parsed_data["instances"]
        elif isinstance(parsed_data, dict):
            scenes_list = [parsed_data]
        elif isinstance(parsed_data, list):
            scenes_list = parsed_data
        else:
            scenes_list = []

        if scenes_list:
            for scene_data in scenes_list:
                output.append({
                    "File Name": f,
                    "Instance Text": scene_data.get("instance", ""),
                    "Reasoning Focalization": scene_data.get("reasoning_focalization", ""),
                    "Label Focalization": scene_data.get("label_focalization", ""),
                    "Reasoning Explicity": scene_data.get("reasoning_explicity", ""),
                    "Label Explicity": scene_data.get("label_explicity", "")
                })
        #Exception for if the output is empty
        else:
            output.append({
                "File Name": f,
                "Instance Text": "No relevant scenes identified.",
                "Reasoning Focalization": "N/A",
                "Label Focalization": "N/A",
                "Reasoning Explicity": "N/A",
                "Label Explicity": "N/A"
            })
    #Failsafe if the LLM output does not match any format
    except Exception as e:
        output.append({
            "File Name": f,
            "Instance Text": f"Error parsing JSON. Raw output: {txt}",
            "Reasoning Focalization": "Error",
            "Label Focalization": "Error",
            "Reasoning Explicity": "Error",
            "Label Explicity": "Error"
        })

df = pd.DataFrame(output)

csv_output_path = "output.csv"
df.to_csv(csv_output_path, index=False)

print(f"\nProcessing complete! Dataframe created and exported to {csv_output_path}")
